# OOPAO + 算法 1 仿真逐步调试 Notebook

对 **beaconless-ao-sim** 的物理仿真流水线（`data/simulate.py`）做逐步可视化调试，每个单元对应算法 1 的一个阶段（参见 `REPORT.md` §1.1-1.4 与论文 DiComo 等，*Opt. Express* 33(15):31010 (2025)，DOI 10.1364/OE.561077）。

湍流屏由 **OOPAO** 提供，参考：

- OOPAO 文档主页：<https://cheritier.github.io/OOPAO/index.html>
- [`Atmosphere`](https://cheritier.github.io/OOPAO/atmosphere.html)：多层 von-Karman 相位屏（基于 `phaseStats.ft_sh_phase_screen`，源自 aotools）。
- [`Telescope`](https://cheritier.github.io/OOPAO/telescope.html)：圆形孔径几何，采样网格 `resolution = N`。
- [`Source`](https://cheritier.github.io/OOPAO/source.html)：点源 + 波长带（`optBand`）；本文用 `optBand="R"`（λ ≈ 650 nm，仅用于 OPD 单位约定，OPD 本身以弧度计算）。
- [`Zernike`](https://cheritier.github.io/OOPAO/zernike.html)：Noll 序 Zernike 多项式基底。
- [Phase Statistics](https://cheritier.github.io/OOPAO/phase_stats.html)：`ft_sh_phase_screen`（OOPAO 内部用，源自 aotools）。

本项目里 OOPAO 的接入点：

- `physics/oopao_backend.py` 中的 `OopaoScreenBackend`：构建一次 `Atmosphere`（参考 r0 = 0.15 m @ 500 nm），用 `Atmosphere.generateNewPhaseScreen(seed)` 抽取每层屏，再以 `(r0_slab / r0_ref)^(5/6)` 把每层振幅缩放到目标 per-slab r0，并中心裁剪到 `N × N`。
- `data/simulate.py`：用 `shared.oopao.make_screens(seed)` 替换 `_make_screens` 里的 aotools 路径（`beam_source == "oopao"`）。

**运行方式**：
```bash
uv run jupyter notebook data/oopao_simulation_debug.ipynb
```
（或 VS Code 直接打开；内核选 `.venv`）。

**调试技巧**：
- 改 `cfg` 后重跑「§1 构建共享状态」单元，使新参数生效。
- 单次 `simulate_sample` 约 5-6 s（含 10 层 OOPAO 屏 + 10 个粗糙面 realization）。
- 每个 step 单元都可视化对应阶段的关键中间量，方便定位数值/几何异常。


## 算法 1 全流程图（来自 `REPORT.md` §1.1）

```mermaid
flowchart TD
    A["聚焦高斯光束<br/>束腰 λL/D"] --> B["10 层 OOPAO<br/>von-Karman 相位屏<br/>间隔 100 m / 1 km"]
    B --> C["分步 FFT 传播<br/>propagation_fft"]
    C --> D["目标面 1 km<br/>散射 / 粗糙表面"]
    D --> E["算法 1 导引信标<br/>衍射极限高斯"]
    E --> F["反向传播至瞳孔<br/>+ 解析抛物面离焦移除"]
    F --> G["共轭信标相位<br/>Φ_beacon"]
    G --> H["Zernike 投影<br/>Φ_Z78"]
    H --> I["CNN 目标<br/>78 阶 Noll 系数"]
    G --> J["倾斜 / 倾斜跟踪"]
    D --> K["3 测量平面成像<br/>-zR / 0 / +zR"]
    K --> L["非相干平均<br/>+ 吸收边界"]
    L --> M["12-bit 逐图像量化"]
    M --> N["输入 3x512x512<br/>uint16/2047"]
    I --> O["训练 CNN1"]
    N --> O
```

下方每个章节对应算法 1 的一个步骤。


## §0. 环境准备 + 配置加载

导入依赖；加载 `config.yaml`；启用 inline 绘图与中文字体。所有后续单元依赖这里的 `cfg`。


In [ ]:
import os, sys, time
import yaml
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

with open(os.path.join(ROOT, 'config.yaml'), encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

mpl.rcParams['figure.dpi'] = 110
mpl.rcParams['image.cmap'] = 'inferno'
for cand in ('Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'PingFang SC'):
    if cand in {f.name for f in mpl.font_manager.fontManager.ttflist}:
        mpl.rcParams['font.sans-serif'] = [cand, 'DejaVu Sans']
        break
mpl.rcParams['axes.unicode_minus'] = False

print('ROOT =', ROOT)
print('beam_source =', cfg['physical']['beam_source'])
print('N =', cfg['physical']['N'], '  L =', cfg['physical']['L'], 'm  lam =', cfg['physical']['wavelength']*1e9, 'nm')


## §1. 构建 `SharedSim`（传播器、Zernike 基底、网格、光束）

`_get_shared(cfg)` 一次性构建整个进程复用、跨样本不变的重计算量：

- `Propagator`（FFT 分步传播器，FFTW 初始化约 1-3 s）
- `ZernikeBasis`（78 阶 Noll 基，伪逆约 1 s）
- 坐标网格 `X/Y/r2`、孔径掩膜 `pupil`、跟踪高斯权重 `G`
- 入瞳光束 `E0`、聚焦相位 `phi_focus`、真空目标面强度 `I_vac`
- 成像几何 `zR_APWS / f_obj / plane_offsets`（公式 9-12 解析解）
- `oopao: OopaoScreenBackend`（仅当 `beam_source == "oopao"`）

**调试点**：800 nm / Cn² = 8.13e-15 / L = 1 km 时 `zR_APWS ≈ 642 m`、`f_obj ≈ 1285 m`，对应 `D/r0 ≈ 7.4`（强湍流）。


In [ ]:
from data.simulate import (
    _get_shared, simulate_sample, _make_screens, _imaging,
    _fom_leg, _beacon_phase_conj, _tracking, _quantize,
)

t0 = time.time()
shared = _get_shared(cfg)
print(f'共享状态构建耗时 {time.time()-t0:.1f} s')

print(f'N = {shared.N}   dx = {shared.dx*1e3:.3f} mm   lam = {shared.lam*1e9:.0f} nm')
print(f'rspot = {shared.rspot*1e2:.1f} cm   focal = {shared.focal:.0f} m')
print(f'zR_APWS = {shared.zR_APWS:.2f} m   f_obj = {shared.f_obj:.2f} m')
print(f'plane_offsets (m) = {np.round(shared.plane_offsets, 2)}')
print('  (plane 0 = f_obj - zR, plane 1 = 焦平面, plane 2 = f_obj + zR)')
print(f'E0 max = {np.abs(shared.E0).max():.3f}   pupil pixels = {int(shared.pupil.sum())}')
print(f'I_vac max = {shared.I_vac.max():.4f}   oopao backend: {shared.oopao is not None}')


### §1.1 可视化：入瞳光束 / 聚焦相位 / 真空目标面

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(np.abs(shared.E0)**2); ax[0].set_title('|E0|^2 入瞳高斯'); ax[0].axis('off')
ax[1].imshow(shared.phi_focus, cmap='twilight', vmin=-np.pi, vmax=np.pi)
ax[1].set_title('phi_focus (rad)'); ax[1].axis('off')
ax[2].imshow(shared.I_vac, vmin=0, vmax=np.percentile(shared.I_vac, 99.9))
ax[2].set_title('I_vac（无湍流目标面）'); ax[2].axis('off')
plt.tight_layout(); plt.show()


## §2. 步骤 A：OOPAO 湍流屏 `Atmosphere` → 10 层 von-Karman 相位屏

对应 OOPAO 文档：

- [`Atmosphere.generateNewPhaseScreen`](https://cheritier.github.io/OOPAO/atmosphere.html)：按 `seed` 重置每层的 `phaseStats.ft_sh_phase_screen`（OOPAO 内部由 aotools 移植），输出 `layer_i.OPD`（弧度，未做 `2π/λ` 转换）。
- [`Atmosphere.initializeAtmosphere`](https://cheritier.github.io/OOPAO/atmosphere.html)：构造每层并（可选）计算协方差矩阵。我们 `compute_covariance=False`，不进入 OOPAO 的 WFS/CL 工具链，只取屏本身。

本项目里 `OopaoScreenBackend` 流程：
1. 构建参考 r0 = 0.15 m @ 500 nm 的 10 层大气；
2. `generateNewPhaseScreen(seed)` 抽 10 层 516×516（含 2 px frozen-flow 边带）；
3. 中心裁剪到 512×512；
4. 每层振幅 × `(r0_slab / r0_ref)^(5/6)`，得到与 aotools 路径等价的 per-slab r0。

**调试点**：每层 OPD 标准差应稳定在 ~1.3 rad（与 aotools ~1.27 rad 比值 ≈ 1.06，落在采样噪声内）。


In [ ]:
SEED = int(cfg['data']['master_seed'])
t0 = time.time()
screens = _make_screens(SEED, cfg, shared)
print(f'屏幕生成耗时 {time.time()-t0:.2f} s   shape = {screens.shape}')

fig, ax = plt.subplots(2, 5, figsize=(14, 5))
for i, a in enumerate(ax.flat):
    if i < screens.shape[0]:
        a.imshow(screens[i], cmap='twilight', vmin=-3, vmax=3)
        a.set_title(f'屏 {i}\nstd={screens[i].std():.2f} rad')
    a.axis('off')
plt.suptitle(f'OOPAO Atmosphere 10 层屏 (seed={SEED})')
plt.tight_layout(); plt.show()

print('每层 OPD std (rad) =', np.round(screens.std(axis=(1,2)), 3))


### §2.1 对比 OOPAO 屏 vs aotools 屏（`beam_source: "aotools"`）

切换 `cfg['physical']['beam_source']` 后重建 `shared`，用相同 seed 比较两条路径的统计量。两者在统计上应等价（per-slab r0 相同）。


In [ ]:
import copy
cfg_alt = copy.deepcopy(cfg)
cfg_alt['physical']['beam_source'] = 'aotools'
shared_alt = _get_shared(cfg_alt)
screens_aotools = _make_screens(SEED, cfg_alt, shared_alt)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, (name, scr) in zip(ax, [('OOPAO', screens), ('aotools', screens_aotools)]):
    a.imshow(scr.std(axis=0), cmap='inferno')
    a.set_title(f'{name}: std across layers\nmean per-layer std = {scr.std(axis=(1,2)).mean():.3f}')
    a.axis('off')
plt.suptitle('两种屏生成路径 — 等价性检查（统计量应近似一致）')
plt.tight_layout(); plt.show()

cfg['physical']['beam_source'] = 'oopao'
shared = _get_shared(cfg)


## §3. 步骤 B+C：聚焦光束前向分步传播 → 湍流下的目标面强度

入瞳场 `E0 e^{i phi_focus}` 经 10 层屏做 `dz = 100 m` 分步传播到 `L = 1000 m` 目标面：`split_step(E, screens, dz)`（`physics/propagation_fft.py`）。这给出 `_fom_leg` 中的 `I_obj_noao`。

**调试点**：真空下 `I_vac` 应是一颗亮的衍射极限核（无湍流）；加入屏后会出现明显的散斑/闪烁。


In [ ]:
phi_focus = shared.phi_focus
E_in = (shared.E0 * np.exp(1j * phi_focus)).astype(np.complex64)
t0 = time.time()
E_noao = shared.prop.split_step(E_in, screens, shared.dz)
I_noao = (np.abs(E_noao)**2).astype(np.float32)
print(f'前向 split_step 耗时 {time.time()-t0:.2f} s')

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(shared.I_vac, vmin=0, vmax=np.percentile(shared.I_vac, 99.9))
ax[0].set_title('真空 I_vac'); ax[0].axis('off')
ax[1].imshow(I_noao, vmin=0, vmax=np.percentile(I_noao, 99.9))
ax[1].set_title('湍流（无 AO）目标面 I_noao'); ax[1].axis('off')
ax[2].imshow(np.log10(I_noao + 1e-12), cmap='inferno')
ax[2].set_title('log10 I_noao（对数标度看闪烁）'); ax[2].axis('off')
plt.tight_layout(); plt.show()


## §4. 步骤 D：算法 1 导引信标反向传播 → `phi_conj`

在目标面放一个**衍射极限小高斯**信标（束腰 `w = λL/D`，避免单像素 δ 的无限带宽伪影），用 `_beacon_phase_conj` 做反向传播：

1. 倒序穿屏 `split_step(E_pt, screens[::-1], -dz)` → 瞳孔处场 `E_back`；
2. **解析移除抛物面离焦** `exp(+i k r²/(2L))`（避免被弱场边缘离群污染）；
3. **强度引导的洪水填充 2D 解卷绕**（numba）→ `phi_unwrapped`；
4. 移除 piston（孔径内均值为零）→ `phi_unwrapped`；
5. 取共轭 → `phi_conj`。

**调试点**：检查 `I_beacon` 是否集中在瞳孔内、`phi_conj` 是否与对流层相位一致。


In [ ]:
t0 = time.time()
phi_conj, I_beacon = _beacon_phase_conj(SEED, cfg, shared, screens)
print(f'信标反向传播 + 解卷绕 + 共轭 耗时 {time.time()-t0:.2f} s')

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(I_beacon, cmap='inferno')
ax[0].set_title('|E_back|^2 瞳孔处信标强度'); ax[0].axis('off')
ax[1].imshow(phi_conj, cmap='twilight', vmin=-3, vmax=3)
ax[1].set_title('phi_conj 共轭信标相位 (rad)'); ax[1].axis('off')
ax[2].imshow(phi_conj - phi_conj[shared.pupil].mean(), cmap='twilight', vmin=-3, vmax=3)
ax[2].set_title('piston-removed'); ax[2].axis('off')
plt.tight_layout(); plt.show()

print(f'phi_conj (pupil 内) std = {phi_conj[shared.pupil].std():.3f} rad   '
      f'min/max = {phi_conj[shared.pupil].min():.2f} / {phi_conj[shared.pupil].max():.2f}')


## §5. 步骤 E：倾斜跟踪 `_tracking`

用高斯权重 `G`（直径与出射光束一致）做 `phi_conj` 的加权梯度平均，得到倾斜斜率 `[a_x, a_y]`，构造线性斜坡 `phi_track = a_x x + a_y y`。这是 `phi_beacon = phi_conj - phi_track` 中要扣除的低阶 tilt。

**调试点**：`phi_track` 应该是平坦的低阶线性相位；`phi_beacon` 应集中在中-高阶 Zernike。


In [ ]:
phi_track, slopes = _tracking(shared, phi_conj)
phi_beacon = phi_conj - phi_track
print(f'tilt slopes [a_x, a_y] = {np.round(slopes, 6)} rad/m')

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(phi_track, cmap='twilight')
ax[0].set_title('phi_track = a_x x + a_y y'); ax[0].axis('off')
ax[1].imshow(phi_beacon, cmap='twilight', vmin=-3, vmax=3)
ax[1].set_title('phi_beacon = phi_conj - phi_track'); ax[1].axis('off')
ax[2].imshow(np.abs(phi_beacon) > np.percentile(np.abs(phi_beacon), 95), cmap='inferno')
ax[2].set_title('|phi_beacon| 上 5% 分位掩膜'); ax[2].axis('off')
plt.tight_layout(); plt.show()


## §6. 步骤 F：78 阶 Zernike 投影 → `labels` 与 `phi_z78`

OOPAO `Zernike`（<https://cheritier.github.io/OOPAO/zernike.html>）提供 Noll 序基底；本项目里 `physics/zernike_aotools.py` 用 aotools 的 `zernike_nm` 实现相同的 Noll 基，并预计算 `M⁺`（伪逆）做 Zernike→phase 与 phase→Zernike 的快速投影。

`labels = M⁺_Z78 · phi_beacon`、`phi_z78 = M_Z78 · labels`。`labels` 即 CNN 训练目标（论文算法 1）。


In [ ]:
zern = shared.zern
labels = zern.phase_to_zernike(phi_beacon)
phi_z78 = zern.zernike_to_phase(labels)
residual = phi_beacon - phi_z78

print(f'labels shape = {labels.shape}   RMS = {np.sqrt(np.mean(labels**2)):.4f} rad')
print(f'phi_z78 std = {phi_z78[shared.pupil].std():.3f}   phi_beacon std = {phi_beacon[shared.pupil].std():.3f}')
print(f'残余 std (pupil 内) = {residual[shared.pupil].std():.4f} rad')

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].bar(np.arange(1, 79), labels); ax[0].set_xlabel('Noll j'); ax[0].set_ylabel('coeff (rad)')
ax[0].set_title('78 阶 Zernike 系数（CNN 目标）'); ax[0].grid(True, alpha=0.3)
ax[1].imshow(phi_z78, cmap='twilight', vmin=-3, vmax=3)
ax[1].set_title('phi_z78 = M · labels'); ax[1].axis('off')
ax[2].imshow(residual, cmap='twilight')
ax[2].set_title('残余（>78 阶，应近似零）'); ax[2].axis('off')
plt.tight_layout(); plt.show()


## §7. 步骤 G：四分支 FOM（公式 6-8 的桶积分）

调用 `_fom_leg` 算四个 FOM（每条腿：前向传播聚焦+修正相位 → `I_obj` → `FOM = nPIB/SIB`）：

- `noao`：仅聚焦（无校正）
- `track`：聚焦 + 倾斜跟踪
- `beacon`：聚焦 + 跟踪 + 共轭信标（理想上界，~0.93）
- `z78`：聚焦 + 跟踪 + 78 阶 Zernike 重构（CNN 的理想目标，~0.88）

`gain = FOM_ML / FOM_track`，`eta = (FOM_ML - FOM_track) / (FOM_z78 - FOM_track)`。


In [ ]:
phases = {
    'noao':   phi_focus,
    'track':  phi_focus + phi_track,
    'beacon': phi_focus + phi_track + phi_beacon,
    'z78':    phi_focus + phi_track + phi_z78,
}
foms = {}
for k, p in phases.items():
    foms[k] = _fom_leg(shared, screens, p)
    print(f'  {k:7s} FOM = {foms[k]:.4f}')

fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
for a, (k, p) in zip(ax, phases.items()):
    E = shared.prop.split_step((shared.E0 * np.exp(1j * p)).astype(np.complex64), screens, shared.dz)
    I = (np.abs(E)**2).astype(np.float32)
    a.imshow(I, vmin=0, vmax=np.percentile(shared.I_vac, 99.9))
    a.contour(shared.bucket_mask.astype(float), colors='cyan', linewidths=0.6)
    a.set_title(f'{k}: FOM={foms[k]:.3f}'); a.axis('off')
plt.suptitle('各分支前向传播 → 目标面强度（青=桶掩膜）')
plt.tight_layout(); plt.show()


## §8. 步骤 H：多平面粗糙面成像（修正后的关键路径）

流程（论文 §2.4，图 2；REPORT.md §1.3）：

1. 仅跟踪条件目标面场 `E_obj_track` → 强度 `I_obj_track`
2. 对 `n_roughness = 10` 个独立粗糙面 realization 散射
3. 反向穿过倒序屏
4. **吸收边界**：乘 `pupil`（移除孔径外场，防 FFT 卷绕点亮边缘）
5. 共轭准直 + 物镜聚焦
6. 传播到 3 个测量平面，**每平面再乘 `pupil`**
7. 逐 realization **强度平均**（非相干成像）

**调试点**：焦平面（plane 1）应为亮核 + 暗晕，中心/边缘强度比 ≥ 10×。


In [ ]:
t0 = time.time()
images, I_obj_track = _imaging(SEED, cfg, shared, screens, phi_track)
print(f'多平面成像耗时 {time.time()-t0:.2f} s   shape = {images.shape}')

N = shared.N
rr = np.sqrt((np.arange(N)[:,None]-N//2)**2 + (np.arange(N)[None,:]-N//2)**2).astype(int)
print('\n每平面 中心/边缘 强度比（焦平面应 >> 1）：')
for p in range(3):
    pl = images[p]
    c = pl[N//2-16:N//2+16, N//2-16:N//2+16].mean()
    e = pl[rr > int(N*0.40)].mean()
    print(f'  plane {p}: center={c:.4f}  edge(r>{int(N*0.40)})={e:.4f}  ratio={c/max(e,1e-9):.1f}x')

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for p, a in enumerate(ax):
    lo, hi = np.percentile(images[p], [5, 99.9])
    a.imshow(np.clip((images[p]-lo)/(hi-lo), 0, 1))
    a.set_title(f'plane {p}  (max={images[p].max():.3f})'); a.axis('off')
plt.suptitle('多平面成像（修正后：吸收边界 + 非相干平均）')
plt.tight_layout(); plt.show()


## §9. 逐图像归一化 + 12-bit 量化

论文图 2：「图像分别进行了归一化处理」。每张图按自身 max 缩放到 12-bit 满量程 (2047)，使焦平面（能量集中）达到满深度。`scale_p` 仅写入 HDF5 作 schema 兼容。


In [ ]:
scale_p = images.max(axis=(1,2))
q = _quantize(images, scale_p)
print('quantized shape =', q.shape, ' dtype =', q.dtype)
print('每平面 max =', [int(q[p].max()) for p in range(3)], '(应均为 2047)')
print('scale_p (raw 每平面 max) =', np.round(scale_p, 4))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for p, a in enumerate(ax):
    a.imshow(q[p], cmap='inferno', vmin=0, vmax=2047)
    a.set_title(f'plane {p} 12-bit uint16 (max={int(q[p].max())})'); a.axis('off')
plt.suptitle('逐图像归一化 → 12-bit 量化结果')
plt.tight_layout(); plt.show()


## §10. 一次完整 `simulate_sample`：所有中间量打包

为方便对照 §2-§9，下面调用 `simulate_sample` 一键拿到 `SimSample`（含 `phase_conj / phase_track / phase_beacon / phase_z78 / images / labels / 四分支 FOM`）。


In [ ]:
t0 = time.time()
sample = simulate_sample(SEED, cfg, shared=shared)
print(f'simulate_sample 耗时 {time.time()-t0:.2f} s')

print(f'  FOM: noao={sample.fom_noao:.4f}  track={sample.fom_track:.4f}  '
      f'beacon={sample.fom_beacon:.4f}  z78={sample.fom_z78:.4f}')
print(f'  labels norm = {np.linalg.norm(sample.labels):.4f}   '
      f'track_slopes = {np.round(sample.track_slopes, 6)}')

fig, ax = plt.subplots(2, 3, figsize=(13, 7))
top = [('phi_conj', sample.phase_conj), ('phi_track', sample.phase_track),
       ('phi_beacon', sample.phase_beacon)]
bot = [('phi_z78', sample.phase_z78),
       ('I_obj_track', sample.I_obj_track),
       ('plane 1 image', sample.images[1])]
for a, (n, im) in zip(ax[0], top):
    a.imshow(im, cmap='twilight', vmin=-3, vmax=3); a.set_title(n); a.axis('off')
for a, (n, im) in zip(ax[1], bot):
    if 'image' in n:
        a.imshow(im, cmap='inferno'); a.axis('off'); a.set_title(n)
    else:
        a.imshow(im, cmap='twilight', vmin=-3, vmax=3); a.set_title(n); a.axis('off')
plt.tight_layout(); plt.show()


## §11. OOPAO 内部：直接调用 `Atmosphere`（脱离 `OopaoScreenBackend` 的最低层验证）

展示 **不**经过 `r0` 缩放/裁剪的 OOPAO 原生屏（参考 r0 = 0.15 m @ 500 nm，516×516 含 2 px frozen-flow 边带），并对比 `OopaoScreenBackend` 处理后的最终屏。


In [ ]:
from physics._oopao_compat import Atmosphere, Telescope, Source
from physics.screens_soapy import compute_r0

phys = cfg['physical']
n = int(phys['n_screens'])
_R0_REF_500 = 0.15
_altitudes = np.linspace(50.0, float(phys['L']) - 50.0, n).tolist()
_frac = [1.0 / n] * n

tel = Telescope(resolution=phys['N'], diameter=float(phys['Dscope']),
                fov=0.0, samplingTime=0.001)
src = Source(optBand='R', magnitude=0.0, display_properties=False)
src * tel
atm = Atmosphere(
    tel,
    r0=_R0_REF_500, L0=float(phys['L0']),
    windSpeed=[10.0]*n, fractionalR0=_frac, windDirection=[0.0]*n,
    altitude=_altitudes, src=src,
)
atm.initializeAtmosphere(tel, compute_covariance=False)

atm.generateNewPhaseScreen(seed=int(SEED))
raw_layers = [np.asarray(getattr(atm, f'layer_{i+1}').OPD) for i in range(n)]
print('OOPAO 层 shape =', raw_layers[0].shape, '(N+4=516, 含 2 px 边带)')

fig, ax = plt.subplots(2, n, figsize=(2.0*n, 4.5))
for i in range(n):
    ax[0, i].imshow(raw_layers[i], cmap='twilight', vmin=-3, vmax=3)
    ax[0, i].set_title(f'layer {i}\nraw {raw_layers[i].shape[0]}px'); ax[0, i].axis('off')
    cropped = raw_layers[i][2:-2, 2:-2]
    ax[1, i].imshow(cropped, cmap='twilight', vmin=-3, vmax=3)
    ax[1, i].set_title(f'cropped {cropped.shape[0]}px'); ax[1, i].axis('off')
plt.suptitle('OOPAO Atmosphere 层（参考 r0=0.15m @ 500nm，未缩放）')
plt.tight_layout(); plt.show()

r0_path = compute_r0(float(phys['wavelength']), float(phys['cn2']), float(phys['L']))
r0_slab = r0_path * n ** (3.0/5.0)
rescale = (r0_slab / _R0_REF_500) ** (5.0/6.0)
print(f'r0_path={r0_path*1e3:.2f} mm  r0_slab={r0_slab*1e3:.2f} mm  rescale={rescale:.3f}')
print(f'screens[0].std() / raw cropped std = {screens[0].std() / raw_layers[0][2:-2,2:-2].std():.3f}  '
      f'(应 ≈ rescale={rescale:.3f})')


## §12. 参数敏感性扫描（Cn² 的影响）

改 `Cn²` 后重建 `shared`，跑一个 sample 看 FOM 各分支的变化。`Cn²` 改变 → `_cfg_key` 变 → 自动重建 `shared`（约 4 s）。


In [ ]:
import copy
results = []
for cn2 in [4e-15, 8.13e-15, 1.6e-14]:
    c = copy.deepcopy(cfg); c['physical']['cn2'] = cn2
    sh = _get_shared(c)
    s = simulate_sample(SEED, c, shared=sh)
    results.append((cn2, s.fom_noao, s.fom_track, s.fom_beacon, s.fom_z78))
    print(f'Cn2={cn2:.1e}: noao={s.fom_noao:.3f} track={s.fom_track:.3f} '
          f'beacon={s.fom_beacon:.3f} z78={s.fom_z78:.3f}')

arr = np.array(results)
plt.figure(figsize=(7,4))
plt.plot(arr[:,0], arr[:,1], 'o-', label='noao')
plt.plot(arr[:,0], arr[:,2], 's-', label='track')
plt.plot(arr[:,0], arr[:,3], '^-', label='beacon')
plt.plot(arr[:,0], arr[:,4], 'd-', label='z78')
plt.xscale('log'); plt.xlabel('Cn^2 [m^-2/3]'); plt.ylabel('FOM')
plt.title('FOM vs Cn^2（湍流强度敏感性）'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()


## §13. 关键 OOPAO 调用速查

| OOPAO 类/函数 | 我们怎么用 | 文档 |
|---|---|---|
| `Atmosphere(tel, r0, L0, windSpeed, fractionalR0, windDirection, altitude, src)` | `OopaoScreenBackend.__init__` | [Atmosphere](https://cheritier.github.io/OOPAO/atmosphere.html) |
| `atm.initializeAtmosphere(tel, compute_covariance=False)` | 跳过 AO/WFS 链路，只取屏 | [Atmosphere](https://cheritier.github.io/OOPAO/atmosphere.html) |
| `atm.generateNewPhaseScreen(seed)` | 按 `seed+layer` 重置 `phaseStats.ft_sh_phase_screen` | [Atmosphere](https://cheritier.github.io/OOPAO/atmosphere.html) |
| `atm.layer_i.OPD` | 取出 `i+1` 层的弧度相位（516×516 含 2 px 边带） | [Atmosphere](https://cheritier.github.io/OOPAO/atmosphere.html) |
| `Telescope(resolution=N, diameter=D, fov=0, samplingTime=...)` | 圆形孔径 + 网格 | [Telescope](https://cheritier.github.io/OOPAO/telescope.html) |
| `Source(optBand, magnitude, display_properties)` + `src * tel` | 绑定波长带，初始化 `src.optical_path` | [Source](https://cheritier.github.io/OOPAO/source.html) |
| `Zernike(...)` | OOPAO 原生 Noll 基；本项目改用 aotools `zernike_nm` 做相同基 | [Zernike](https://cheritier.github.io/OOPAO/zernike.html) |
| `OOPAO.phaseStats.ft_sh_phase_screen` | OOPAO 屏生成核心（源自 aotools） | [Phase Stats](https://cheritier.github.io/OOPAO/phase_stats.html) |

## 参考

- 论文：G. P. DiComo 等, *Opt. Express* 33(15):31010 (2025), DOI 10.1364/OE.561077
- OOPAO 文档：<https://cheritier.github.io/OOPAO/index.html>
- 本仓库报告：`REPORT.md` §1（物理模拟与数据生成）
- 相关源代码：`data/simulate.py`, `physics/oopao_backend.py`, `physics/propagation_fft.py`, `physics/zernike_aotools.py`, `physics/scattering.py`
